# Demand settings

Prepares the `demand_profile.csv` that interruption analysis attaches to a saved topology network. `00_build_network.ipynb` does not need demand to build `base` or `inferred`.

Two options:

1. **Reviewed (preferred):** place a dated `demand_profile.csv` in `data/1-processed/energy/collaborator/` — a timestamp column plus either one `demand_mw` column or one column per `bus_id`.
2. **Provisional:** derive a one-snapshot profile from `monthly_peak_demand_mw.csv` for pipeline tests. This is not observed demand and is labelled in `demand_profile_metadata.json`.


In [ ]:
import json

import pandas as pd

from mu_star_energy.network_source import provisional_demand_profile
from mu_star_energy.paths import processed_energy_dir
from mu_star_energy.runner import read_time_series_csv

COLLABORATOR_DIR = processed_energy_dir() / "collaborator"
USE_PROVISIONAL_MONTHLY_PEAK = False

demand_path = COLLABORATOR_DIR / "demand_profile.csv"
metadata_path = COLLABORATOR_DIR / "demand_profile_metadata.json"

if demand_path.exists():
    demand_profile = read_time_series_csv(demand_path, label="demand_profile")
    source_label = "reviewed"
elif USE_PROVISIONAL_MONTHLY_PEAK:
    demand_profile = provisional_demand_profile(COLLABORATOR_DIR)
    demand_profile.rename_axis("timestamp").reset_index().to_csv(demand_path, index=False)
    metadata_path.write_text(
        json.dumps(
            {
                "source": "monthly_peak_demand_mw.csv",
                "provisional": True,
                "notes": "One-snapshot profile for pipeline tests; replace with reviewed demand.",
            },
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )
    source_label = "provisional monthly peak"
else:
    demand_profile = None
    source_label = "missing"

print("Demand profile source:", source_label)
print("Demand profile path:", demand_path)

if demand_profile is not None:
    display(pd.Series({
        "snapshots": len(demand_profile),
        "first_timestamp": str(demand_profile.index.min()),
        "last_timestamp": str(demand_profile.index.max()),
        "columns": ", ".join(demand_profile.columns.astype(str)),
        "peak_mw": float(demand_profile.max().max()),
    }, name="value").to_frame())
